# 📊 Modelo de Liquidación de Inversiones - Notebook de Desarrollo

> **Estado**: 🚧 EN DESARROLLO - NO PRODUCTIVO  
> **Última actualización**: 2026-02-03  
> **Progreso**: 19/27 pasos completados (70%)

---

## 📋 Índice

| Sección | Descripción | Estado |
|---------|-------------|--------|
| **A. Setup y Configuración** | Carga de datos, limpieza de tablas base | ✅ |
| **B. Pipeline por Instrumento** | Pasos 1-19: Flujos de liquidación | ✅ |
| **C. Tabla Final de Inversiones** | Pasos 20-21: UNION de flujos | 🔄 |
| **D. Integración Tabla Desarrollo** | Pasos 22-27: Consolidación final | ⬜ |
| **E. Validación y Comparación** | Comparación vs Access | ✅ |

---

## 📁 Archivos de Referencia

- **[PLAN_MEJORA_NOTEBOOK.md](./PLAN_MEJORA_NOTEBOOK.md)** - Plan de restructuración
- **[RF_PLI_044e_analisis.md](./RF_PLI_044e_Modelo_Inversiones_Tabla_Final_analisis.md)** - Análisis paso 21
- **[analisis_pasos_22_27.md](./analisis_pasos_22_27_tabla_desarrollo.md)** - Análisis pasos 22-27
- **[PLAN_IMPLEMENTACION.md](./PLAN_IMPLEMENTACION.md)** - Plan de refactorización completo

---

## 🔢 Resumen de Pasos Access → Python

| Paso | Query Access | Función Python | Estado |
|------|--------------|----------------|--------|
| 1-7 | RF_PLI_001 a 007 | `generar_flujo_liquidacion_instrumento('GobCLP')` | ✅ |
| 8-14 | RF_PLI_008 a 014 | `generar_flujo_liquidacion_instrumento('GobCLF')` | ✅ |
| 15-19 | RF_PLI_015 a 042 | Pipelines DPF, DPR, BBC, LCH | ✅ |
| 20 | RF_PLI_045_Precios_Dia | `generar_precios_dia()` | 🔄 Simple |
| 21 | RF_PLI_044e_Tabla_Final | `generar_tabla_final_inversiones()` | ⬜ Complejo |
| 22 | RF_PLI_047_Limpia | N/A (Python no necesita) | ✅ |
| 23-26 | RF_PLI_048* | `generar_tabla_desarrollo_completa()` | ⬜ |
| 27 | RF_PLI_050_Excel | `formatear_para_excel()` | ⬜ |

In [2]:
"""
draft traduccion Modelo de Liquidación de inversiones desde Access/Excel a Python
========================================



Autor: Modelos & Metodologías
Fecha: 2026-02
"""

import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import yaml
from pathlib import Path
import sys
import bfa_cl_utilidades as ut
import os
import pathlib
import dotenv
import datetime
import pickle
import helpers as ml_inv_helpers

fecha_proceso = 20260115

# encontremos el path del notebook actual
notebook_path = pathlib.Path().resolve()
BASE_DIR = notebook_path
# buscamos el directorio raíz del proyecto (donde está el .git)
while not (BASE_DIR / '.git').exists():
    BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
# Importación de módulos internos
from config import config_rutas as cr  # Configuración de rutas del proyecto


# Carga de configuración desde archivo YAML
with open(cr.CONFIG / 'config_rutas_ext_y_archivos.yaml', 'r') as file:
    config_ext = yaml.safe_load(file)

# PROCESOS_DIARIOS_MODELOS: antiguo folder de desarrollo de modelos diarios
# vamos a definir su path, que es ~/code/PROCESOS_DIARIOS_MODELOS

procesos_diarios_modelos_path = BASE_DIR.parent / 'PROCESOS_DIARIOS_MODELOS'

data_path = procesos_diarios_modelos_path / 'data' /'external' / 'ml_inversiones'


accdb_path = data_path/"RF_Gener_Modelo_inversiones_20260115_local.accdb"
df_secuencia = pd.read_csv(data_path / "modelo_inversiones_completo_secuencia.csv",sep=";",)
df_maestro = pd.read_csv(data_path / "maestro_inversiones_funciones_completas.csv",sep=";",)


archivos_pickle = list(data_path.glob(f"tablas_linkeadas_ml_inversiones_{fecha_proceso}_*.pkl"))
if archivos_pickle:
    archivo_reciente = max(archivos_pickle, key=os.path.getctime)
    print(f"Pickle encontrado: {archivo_reciente.name}")
    
    with open(archivo_reciente, "rb") as f:
        tablas_actual = pickle.load(f)
    
    print(f"\nTablas en pickle ({len(tablas_actual)}):")
    for nombre, df in tablas_actual.items():
        print(f"  - {nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")
else:
    print("No se encontró pickle")

# Leyendo links a tablas de otros archivos que usa el access

df_links = pd.read_csv(data_path / "links_ml_inversiones.csv",sep=";",)
# Leyendo tablas del access

df_tablas = pd.read_csv(data_path / "RF_Gener_Modelo_Inversiones_prod" / "RF_Gener_Modelo_Inversiones_tables.csv",sep=";",)        
# leyendo queries con dependencias del access
df_queries = pd.read_csv(data_path / "RF_Gener_Modelo_Inversiones_20251121_local" / "RF_Gener_Modelo_inversiones_20251121_local_all_queries.csv",sep=";",)

queries_raw = pd.read_csv(data_path / "RF_Gener_Modelo_Inversiones_20251121_local" / "RF_Gener_Modelo_inversiones_20251121_local_queries.csv",sep=";",)


tablas = ml_inv_helpers.check_pickle_tablas_linkeadas(fecha_proceso, data_path,df_links)        
tablas.keys()


Pickle encontrado: tablas_linkeadas_ml_inversiones_20260115_20260119_101906.pkl

Tablas en pickle (12):
  - FPL: 10 filas, 5 columnas
  - RF_base_Completa_Hist_Input: 414802 filas, 44 columnas
  - RF_Base_Diaria_Precios: 455659 filas, 24 columnas
  - RF_Base_Diaria_Precios_: 244010 filas, 24 columnas
  - RF_BD_Gestion_RM: 146955 filas, 23 columnas
  - RF_Cartera_RtaFija_Hist: 218556 filas, 20 columnas
  - RF_FactCLF_Banc: 1350 filas, 4 columnas
  - RF_FactCLF_Gob: 1350 filas, 4 columnas
  - RF_FactCLP_Banc: 1260 filas, 4 columnas
  - RF_FactCLP_Gob: 1260 filas, 4 columnas
  - RF_Fecha_Proceso_Carteras: 1 filas, 1 columnas
  - RF_MontosLiq: 8 filas, 5 columnas
Cargando tablas linkeadas desde pickle: C:\Users\vlandaetat\code\PROCESOS_DIARIOS_MODELOS\data\external\ml_inversiones\tablas_linkeadas_ml_inversiones_20260115_20260119_101906.pkl


dict_keys(['FPL', 'RF_base_Completa_Hist_Input', 'RF_Base_Diaria_Precios', 'RF_Base_Diaria_Precios_', 'RF_BD_Gestion_RM', 'RF_Cartera_RtaFija_Hist', 'RF_FactCLF_Banc', 'RF_FactCLF_Gob', 'RF_FactCLP_Banc', 'RF_FactCLP_Gob', 'RF_Fecha_Proceso_Carteras', 'RF_MontosLiq'])

In [3]:
from helpers import genera_cartera_inv_001,generar_cartera_instrumento, \
      generar_monto_total_instrumento, generar_cartera_pond 

In [4]:
tablas['RF_base_Completa_Hist'] = ml_inv_helpers.genera_tabla_RF_base_Completa_Hist(tablas['RF_base_Completa_Hist_Input'], fecha_proceso)

tabla_fecha_proceso = pd.DataFrame({'Fecha':[pd.to_datetime(str(fecha_proceso), format="%Y%m%d")]})
queries = {}
queries['RF_PLI_001_CarteraInv'] = genera_cartera_inv_001(
    df_base=tablas['RF_base_Completa_Hist'],
     df_fecha=tabla_fecha_proceso,
    verbose=True
)

COLUMNAS_SALIDA = [
    'Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_Pro', 'Cod_Sub_Pro',
    'Nemotecnico', 'Instrumento', 'VP_Cap_Amort', 'VP_Int_Total', 'Dias_Vcto'
]

INSTRUMENTOS = {'GobCLP':['BCP','BTP','PDB'], 'GobCLF':['BCU','BTU'],
                'DPF':['DPF'],'DPR':['DPR'], 
                'LCH':['LCH','BBC'],'BBC':['BBC'],
                               }

queries['RF_PLI_002_CarteraGobCLP'] = generar_cartera_instrumento(
    queries['RF_PLI_001_CarteraInv'],
    COLUMNAS_SALIDA,
    INSTRUMENTOS['GobCLP'],
    'GobCLP'
)

queries['RF_PLI_003_CarteraGobCLP_MonTotal'] = generar_monto_total_instrumento(
    queries['RF_PLI_002_CarteraGobCLP'],
    ['Fec_Pro', 'Cod_Pro', 'Moneda'],
    ['VP_Cap_Amort', 'VP_Int_Total'],
    'GobCLP'
)
queries['RF_PLI_004_CarteraGobCLP_Pond'] = generar_cartera_pond(
    queries['RF_PLI_002_CarteraGobCLP'],
    queries['RF_PLI_003_CarteraGobCLP_MonTotal'],
    'RF_CarteraGobCLP_Pond'
)


PASO 01b: RF_PLI_001_CarteraInv
Registros entrada: 414,802

[JOIN] Filtro fecha proceso = 2026-01-15
  Registros que cumplen: 51,748

[WHERE 1] Nemotecnico NO empieza con 'LCH'
  Registros que cumplen: 408,778

[WHERE 2] Cod_Pro es inversión financiera
  Registros que cumplen: 11,718

[WHERE 3] Cod_Sub_Pro termina en 'Disp', 'Disp_Liq' o 'MUTUOS'
  Registros que cumplen: 11,493

[WHERE 4] Clasificacion_Contable <> 'HTM'
  Registros que cumplen: 408,578

[WHERE FINAL] Todos los filtros combinados (AND)
  Registros que cumplen: 675

RESULTADO FINAL
  Registros entrada: 414,802
  Registros salida:  675

  Distribución Cod_Pro:
Cod_Pro
Inversion Financiera Privado    369
Inversion Financiera Publico    306

  Distribución Instrumento:
Instrumento
BBC    356
BTP    232
BTU     68
PDB      6
DPF      6
DPR      5
OTR      2
Cartera GobCLP: 238 registros después de filtrar por instrumento ['BCP', 'BTP', 'PDB']
Monto total GobCLP generado: 1 registros después de agrupar por ['Fec_Pro', 'Cod_P

In [5]:
# =============================================================================
# RECARGAR HELPERS PARA OBTENER LAS NUEVAS FUNCIONES
# =============================================================================
import importlib
import helpers
importlib.reload(helpers)

# Verificar que existe la nueva función
print("Funciones disponibles en helpers:")
funciones_relevantes = [f for f in dir(helpers) if 'flujo' in f.lower() or 'configuracion' in f.lower()]
for func in funciones_relevantes:
    print(f" - {func}")

# Verificar configuración de instrumentos
print("\n" + "="*50)
print("CONFIGURACIÓN DE INSTRUMENTOS DISPONIBLES:")
print("="*50)
for inst, config in helpers.CONFIGURACION_INSTRUMENTOS.items():
    print(f"\n{inst}:")
    print(f"  - Códigos disp: {config['codigos_disp']}")
    print(f"  - Códigos pacto: {config['codigos_pacto']}")
    print(f"  - Tabla factores: {config['tabla_factores']}")

Funciones disponibles en helpers:
 - CONFIGURACION_INSTRUMENTOS
 - calcular_flujo_liquidacion
 - generar_flujo_liquidacion_instrumento

CONFIGURACIÓN DE INSTRUMENTOS DISPONIBLES:

GobCLP:
  - Códigos disp: ['BCP', 'BTP', 'PDB']
  - Códigos pacto: ['BCP', 'BTP', 'PDB']
  - Tabla factores: RF_FactCLP_Gob

GobCLF:
  - Códigos disp: ['BCU', 'BTU']
  - Códigos pacto: ['BCU', 'BTU', 'CER']
  - Tabla factores: RF_FactCLF_Gob

DPF:
  - Códigos disp: ['DPF']
  - Códigos pacto: ['DPF', 'FFM']
  - Tabla factores: RF_FactCLP_Banc

DPR:
  - Códigos disp: ['DPR']
  - Códigos pacto: ['DPR']
  - Tabla factores: RF_FactCLF_Banc

BBC:
  - Códigos disp: ['BBC']
  - Códigos pacto: ['BBC']
  - Tabla factores: RF_FactCLP_Banc

LCH:
  - Códigos disp: ['LCH', 'BBC']
  - Códigos pacto: ['LCH', 'BBC']
  - Tabla factores: RF_FactCLF_Banc


In [6]:
# TODO: agregar DPX (en USD) con los parámetros de DPR, pero con montos a liquidar en dólares. 


---
# 🔧 SECCIÓN A: Setup y Configuración

## A.1 Limpieza de Tablas Base

Verificamos y limpiamos la estructura de las tablas linkeadas necesarias:
- **FPL**: Factores de haircut por instrumento
- **RF_MontosLiq**: Montos a liquidar por instrumento

In [7]:
# =============================================================================
# FASE 0b: Limpieza de Tablas Base
# =============================================================================

print("="*70)
print("LIMPIEZA DE TABLAS BASE")
print("="*70)

# 1. FPL - Limpiar columnas basura y filas NaN
print("\n[1] Limpiando FPL...")
tablas['FPL'] = tablas['FPL'][['Instrumento', 'Haircut']].dropna()
print(f"    FPL limpio: {len(tablas['FPL'])} registros")
display(tablas['FPL'])

# 2. RF_MontosLiq - Re-estructurar (headers en fila 0, col 0 vacía)
print("\n[2] Re-estructurando RF_MontosLiq...")
df_montos_raw = tablas['RF_MontosLiq'].copy()
# Tomar desde la segunda columna (ignorar col 0 vacía)
df_montos_clean = df_montos_raw.iloc[1:, 1:].copy()  # Desde fila 1, desde col 1
# Usar la primera fila como headers
df_montos_clean.columns = df_montos_raw.iloc[0, 1:].values
# Resetear índice
df_montos_clean = df_montos_clean.reset_index(drop=True)
# Convertir columnas numéricas
for col in ['Monto Mercado', '% participacion', 'Monto a Liquidar']:
    df_montos_clean[col] = pd.to_numeric(df_montos_clean[col], errors='coerce')
# Guardar tabla limpia
tablas['RF_MontosLiq'] = df_montos_clean
print(f"    RF_MontosLiq limpio: {len(tablas['RF_MontosLiq'])} registros")
display(tablas['RF_MontosLiq'])

print("\n" + "="*70)
print("✓ FASE 0 COMPLETADA - Tablas listas para usar")
print("="*70)

LIMPIEZA DE TABLAS BASE

[1] Limpiando FPL...
    FPL limpio: 7 registros


,Instrumento,Haircut
0,Gobierno CLP,0.002862
1,Gobierno CLF,0.006937
2,DPF,0.004804
3,DPR,0.001980
4,Corporativo CLP,0.012151
5,Corporativo CLF,0.007283
6,LCHR,0.010844



[2] Re-estructurando RF_MontosLiq...
    RF_MontosLiq limpio: 7 registros


,Instrumento,Monto Mercado,% participacion,Monto a Liquidar
0,Gobierno CLP,9.762388e+11,0.113718,1.110155e+11
1,Gobierno CLF,3.114182e+06,0.236630,7.369081e+05
2,DPF,3.030708e+11,0.090793,2.751656e+10
3,DPR,4.086030e+05,0.264012,1.078761e+05
4,Corporativo CLP,1.285508e+10,0.131426,1.689491e+09
5,Corporativo CLF,3.820034e+06,0.136548,5.216180e+05
6,LCHR,1.655843e+04,0.136548,0.000000e+00



✓ FASE 0 COMPLETADA - Tablas listas para usar


---
# 📊 SECCIÓN B: Pipeline por Instrumento (Pasos 1-19)

## B.1 Rama PACTOS

Generamos las queries de la rama de pactos que se usarán para calcular el haircut:

| Query | Descripción | SQL Equivalente |
|-------|-------------|-----------------|
| `RF_PLI_001d_CarteraInv_Pcto` | Cartera de inversiones con pactos | INNER JOIN con fecha, filtro Pcto |
| `RF_PLI_002_CarteraGobCLP_Pacto` | Filtrar solo GobCLP | WHERE Instrumento IN ('BCP','BTP','PDB') |
| `RF_PLI_003b_GobCLP_MontoPlazo_Pacto` | Monto por plazo de pacto | GROUP BY Dias_Pacto |

In [8]:
# =============================================================================
# FASE 1: RAMA PACTOS
# =============================================================================
# Recargamos helpers para usar las nuevas funciones
import importlib
import helpers as ml_inv_helpers
importlib.reload(ml_inv_helpers)

from helpers import genera_cartera_inv_pacto, generar_monto_plazo_pacto

# -----------------------------------------------------------------------------
# 1.1 RF_PLI_001d_CarteraInv_Pcto - Cartera de inversiones con pactos
# -----------------------------------------------------------------------------
queries['RF_PLI_001d_CarteraInv_Pcto'] = genera_cartera_inv_pacto(
    df_base=tablas['RF_base_Completa_Hist'],
    df_fecha=tabla_fecha_proceso,
    verbose=True
)

print("\nPrimeras filas:")
display(queries['RF_PLI_001d_CarteraInv_Pcto'].head())


FASE 1.1: RF_PLI_001d_CarteraInv_Pcto (Cartera Pactos)
Registros entrada: 414,802

[JOIN] Filtro fecha proceso = 2026-01-15
  Registros que cumplen: 51,748

[WHERE 1] Cod_Pro es inversión financiera
  Registros que cumplen: 11,718

[WHERE 2] Cod_Sub_Pro termina en 'Pcto' o 'Pcto_Liq'
  Registros que cumplen: 0

[WHERE FINAL] Todos los filtros combinados (AND)
  Registros que cumplen: 0

RESULTADO: 0 registros generados
Columnas: ['Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_Pro', 'Cod_Sub_Pro', 'Nemotecnico', 'Instrumento', 'VP_Cap_Amort', 'VP_Int_Total', 'Dias_Vcto', 'Dias_Pacto']

Primeras filas:


,Fec_Pro,Cod_Emp,Moneda,Cod_Pro,Cod_Sub_Pro,Nemotecnico,Instrumento,VP_Cap_Amort,VP_Int_Total,Dias_Vcto,Dias_Pacto


In [9]:
# -----------------------------------------------------------------------------
# 1.2 RF_PLI_002_CarteraGobCLP_Pacto - Filtrar por instrumento GobCLP
# -----------------------------------------------------------------------------
# Reutilizamos generar_cartera_instrumento pero con columnas que incluyen Dias_Pacto

COLUMNAS_SALIDA_PACTO = [
    'Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_Pro', 'Cod_Sub_Pro',
    'Nemotecnico', 'Instrumento', 'VP_Cap_Amort', 'VP_Int_Total', 
    'Dias_Vcto', 'Dias_Pacto'
]

queries['RF_PLI_002_CarteraGobCLP_Pacto'] = generar_cartera_instrumento(
    queries['RF_PLI_001d_CarteraInv_Pcto'],
    COLUMNAS_SALIDA_PACTO,
    INSTRUMENTOS['GobCLP'],  # ['BCP', 'BTP', 'PDB']
    'GobCLP_Pacto'
)

print("\nPrimeras filas:")
display(queries['RF_PLI_002_CarteraGobCLP_Pacto'].head())


# -----------------------------------------------------------------------------
# 1.3 RF_PLI_003b_GobCLP_MontoPlazo_Pacto - Monto por plazo de pacto
# -----------------------------------------------------------------------------
queries['RF_PLI_003b_GobCLP_MontoPlazo_Pacto'] = generar_monto_plazo_pacto(
    queries['RF_PLI_002_CarteraGobCLP_Pacto'],
    verbose=True
)

print("\nResultado completo:")
display(queries['RF_PLI_003b_GobCLP_MontoPlazo_Pacto'])

print("\n" + "="*70)
print("✓ FASE 1 COMPLETADA - Rama PACTOS lista")
print("="*70)
print(f"\nQueries generadas:")
print(f"  - RF_PLI_001d_CarteraInv_Pcto: {len(queries['RF_PLI_001d_CarteraInv_Pcto']):,} registros")
print(f"  - RF_PLI_002_CarteraGobCLP_Pacto: {len(queries['RF_PLI_002_CarteraGobCLP_Pacto']):,} registros")
print(f"  - RF_PLI_003b_GobCLP_MontoPlazo_Pacto: {len(queries['RF_PLI_003b_GobCLP_MontoPlazo_Pacto']):,} registros")

Cartera GobCLP_Pacto: 0 registros después de filtrar por instrumento ['BCP', 'BTP', 'PDB']

Primeras filas:


,Fec_Pro,Cod_Emp,Moneda,Cod_Pro,Cod_Sub_Pro,Nemotecnico,Instrumento,VP_Cap_Amort,VP_Int_Total,Dias_Vcto,Dias_Pacto



Resultado completo:


,Dias_Pacto,Monto



✓ FASE 1 COMPLETADA - Rama PACTOS lista

Queries generadas:
  - RF_PLI_001d_CarteraInv_Pcto: 0 registros
  - RF_PLI_002_CarteraGobCLP_Pacto: 0 registros
  - RF_PLI_003b_GobCLP_MontoPlazo_Pacto: 0 registros


---
## B.2 Rama HAIRCUT

Generamos las queries de haircut que determinan qué porcentaje del portafolio se puede liquidar cada día:

| Query | Descripción | Lógica Clave |
|-------|-------------|--------------|
| `RF_PLI_005_CarteraHC` | Cartera con factores de haircut | CROSS JOIN con factores, MAX(Factor, Haircut) |
| `RF_PLI_006_Haircut_Dia` | Haircut agregado por día | GROUP BY Dia, SUM(FactorPond) |
| `RF_PLI_006b_Haircut_Dia` | Haircut con día de semana | Agrega DiaSem para regla fin de semana |

In [10]:
# =============================================================================
# FASE 2: RAMA HAIRCUT
# =============================================================================
# Recargamos helpers para usar las nuevas funciones
importlib.reload(ml_inv_helpers)

from helpers import (
    generar_cartera_haircut, 
    generar_haircut_dia, 
    agregar_dia_semana,
    combinar_haircut_con_pactos,
    filtrar_monto_liquidar
)

# -----------------------------------------------------------------------------
# 2.1 RF_PLI_005_CarteraHC - Cartera con factores de haircut
# -----------------------------------------------------------------------------
queries['RF_PLI_005_CarteraHC'] = generar_cartera_haircut(
    df_cartera_pond=queries['RF_PLI_004_CarteraGobCLP_Pond'],
    df_factores=tablas['RF_FactCLP_Gob'],
    df_fpl=tablas['FPL'],
    filtro_instrumento='Gobierno CLP',
    verbose=True
)

print("\nPrimeras filas:")
display(queries['RF_PLI_005_CarteraHC'].head())


FASE 2.1: RF_PLI_0XX_CarteraHC (Cartera con Haircut)
Registros cartera entrada: 238
Registros factores: 1,260

[FPL] Haircut para '['Gobierno CLP']': 0.002862

[JOIN] Registros después de JOIN con factores: 21,420

[CALC] FactorPond calculado usando MAX(Factor, 0.002862)
  Factor min: 0.000933
  Factor max: 0.059951

RESULTADO: 21,420 registros generados

Primeras filas:


,Fec_Pro,Cod_Emp,Moneda,Cod_Pro,Cod_Sub_Pro,Nemotecnico,Instrumento,VP_Cap_Amort,VP_Int_Total,Dias_Vcto,Ponderador,Dia,Factor,FactorPond
0,2026-01-15,1.0,CLP,Inversion Financiera Publico,Inversion Financiera Publico_TGR_Disp,BTP0581029,BTP,0.0,8.612043e+07,76.0,0.000137,1,0.000933,3.933746e-07
1,2026-01-15,1.0,CLP,Inversion Financiera Publico,Inversion Financiera Publico_TGR_Disp,BTP0581029,BTP,0.0,8.612043e+07,76.0,0.000137,2,0.001210,3.933746e-07
2,2026-01-15,1.0,CLP,Inversion Financiera Publico,Inversion Financiera Publico_TGR_Disp,BTP0581029,BTP,0.0,8.612043e+07,76.0,0.000137,3,0.001487,3.933746e-07
3,2026-01-15,1.0,CLP,Inversion Financiera Publico,Inversion Financiera Publico_TGR_Disp,BTP0581029,BTP,0.0,8.612043e+07,76.0,0.000137,4,0.001764,3.933746e-07
4,2026-01-15,1.0,CLP,Inversion Financiera Publico,Inversion Financiera Publico_TGR_Disp,BTP0581029,BTP,0.0,8.612043e+07,76.0,0.000137,5,0.002041,3.933746e-07


In [11]:
# -----------------------------------------------------------------------------
# 2.2 RF_PLI_006_Haircut_Dia - Haircut agregado por día
# -----------------------------------------------------------------------------
queries['RF_PLI_006_Haircut_Dia'] = generar_haircut_dia(
    queries['RF_PLI_005_CarteraHC'],
    verbose=True
)

print("\nPrimeras filas:")
display(queries['RF_PLI_006_Haircut_Dia'].head(10))


--------------------------------------------------
FASE 2.2: RF_PLI_0XX_Haircut_Dia
--------------------------------------------------
Registros entrada: 21,420
Registros salida: 90
Días: 1 a 90

Primeras filas:


,Dia,Haircut
0,1,0.004047
1,2,0.004205
2,3,0.004362
3,4,0.004520
4,5,0.004679
5,6,0.004838
6,7,0.004997
7,8,0.005254
8,9,0.005510
9,10,0.005766


In [12]:
# -----------------------------------------------------------------------------
# 2.3 RF_PLI_006b_Haircut_Dia - Haircut con día de semana
# -----------------------------------------------------------------------------
queries['RF_PLI_006b_Haircut_Dia'] = agregar_dia_semana(
    queries['RF_PLI_006_Haircut_Dia'],
    fecha_proceso=fecha_proceso,
    verbose=True
)

print("\nPrimeras filas (DiaSem: 1=Lun, ..., 7=Dom):")
display(queries['RF_PLI_006b_Haircut_Dia'].head(10))

print("\n" + "="*70)
print("✓ FASE 2 COMPLETADA - Rama HAIRCUT lista")
print("="*70)


--------------------------------------------------
FASE 2.3: RF_PLI_0XXb_Haircut_Dia (con día semana)
--------------------------------------------------
Registros entrada: 90
Fecha proceso: 2026-01-15 (Jue)
Registros salida: 90

Primeras filas (DiaSem: 1=Lun, ..., 7=Dom):


,Dia,DiaSem,Haircut
0,1,5,0.004047
1,2,6,0.004205
2,3,7,0.004362
3,4,1,0.004520
4,5,2,0.004679
5,6,3,0.004838
6,7,4,0.004997
7,8,5,0.005254
8,9,6,0.005510
9,10,7,0.005766



✓ FASE 2 COMPLETADA - Rama HAIRCUT lista


---
## B.3 Combinación Haircut + Pactos

Combinamos la rama de haircut con los montos de pactos:

**Query**: `RF_PLI_006c_Haircut_Dia_Pcto`

**SQL Original**:
```sql
SELECT Haircut_Dia.*, MontoPlazo_Pacto.Monto_Pacto
FROM Haircut_Dia LEFT JOIN MontoPlazo_Pacto 
ON Haircut_Dia.Dia = MontoPlazo_Pacto.Dias_Pacto
```

In [13]:
# =============================================================================
# FASE 3: COMBINACIÓN HAIRCUT + PACTOS
# =============================================================================
queries['RF_PLI_006c_Haircut_Dia_Pcto'] = combinar_haircut_con_pactos(
    df_haircut_dia_sem=queries['RF_PLI_006b_Haircut_Dia'],
    df_monto_plazo_pacto=queries['RF_PLI_003b_GobCLP_MontoPlazo_Pacto'],
    verbose=True
)

print("\nPrimeras filas:")
display(queries['RF_PLI_006c_Haircut_Dia_Pcto'].head(10))


FASE 3: RF_PLI_0XXc_Haircut_Dia_Pcto (Haircut + Pactos)
Registros haircut: 90
Registros pactos: 0

Registros salida: 90
Días con pactos: 0

Primeras filas:


,Dia,DiaSem,Haircut,Monto_Pacto
0,1,5,0.004047,0.0
1,2,6,0.004205,0.0
2,3,7,0.004362,0.0
3,4,1,0.004520,0.0
4,5,2,0.004679,0.0
5,6,3,0.004838,0.0
6,7,4,0.004997,0.0
7,8,5,0.005254,0.0
8,9,6,0.005510,0.0
9,10,7,0.005766,0.0


---
## B.4 Monto a Liquidar

Obtenemos el monto máximo a liquidar por día para el instrumento desde la tabla `RF_MontosLiq`:

**Query**: `RF_PLI_007_MontoLiquidar`

Este monto representa el **límite diario** de liquidación definido por política de riesgo.

In [14]:
# =============================================================================
# FASE 4: MONTO A LIQUIDAR
# =============================================================================
queries['RF_PLI_007_MontoLiquidar'] = filtrar_monto_liquidar(
    df_montos_liq=tablas['RF_MontosLiq'],
    instrumento='Gobierno CLP',
    verbose=True
)

display(queries['RF_PLI_007_MontoLiquidar'])


--------------------------------------------------
FASE 4: RF_PLI_0XX_MontoLiquidar (Gobierno CLP)
--------------------------------------------------
Monto a liquidar: 111,015,499,404.70


,Instrumento,Monto Mercado,% participacion,Monto a Liquidar
0,Gobierno CLP,9.762388e+11,0.113718,1.110155e+11


---
## B.5 Integración - Función Principal monto_liq_gob_clp

Ejecutamos la función principal `monto_liq_gob_clp()` (ahora deprecada, usar `calcular_flujo_liquidacion()`) que:

1. Toma el monto total de la cartera
2. Aplica haircut exponencial decreciente día a día
3. Respeta reglas de fin de semana (no liquidar sábados/domingos)
4. Devuelve el flujo de liquidación proyectado

**Inputs:**
- `RF_PLI_003_CarteraGobCLP_MonTotal` - Monto total de cartera
- `RF_PLI_006c_Haircut_Dia_Pcto` - Haircut + pactos
- `RF_PLI_007_MontoLiquidar` - Límite diario

In [15]:
# =============================================================================
# FASE 5: INTEGRACIÓN - EJECUTAR FUNCIÓN PRINCIPAL
# =============================================================================
# Recargar helpers con la corrección
importlib.reload(ml_inv_helpers)
from helpers import monto_liq_gob_clp

print("="*70)
print("FASE 5: INTEGRACIÓN - Ejecutando monto_liq_gob_clp()")
print("="*70)

print("\nInputs:")
print(f"  1. RF_PLI_003_CarteraGobCLP_MonTotal: {len(queries['RF_PLI_003_CarteraGobCLP_MonTotal'])} registros")
print(f"     VP_Flujo = {queries['RF_PLI_003_CarteraGobCLP_MonTotal']['VP_Flujo'].sum():,.2f}")
print(f"  2. RF_PLI_006c_Haircut_Dia_Pcto: {len(queries['RF_PLI_006c_Haircut_Dia_Pcto'])} registros")
print(f"  3. RF_PLI_007_MontoLiquidar: {len(queries['RF_PLI_007_MontoLiquidar'])} registros")
print(f"     Monto a Liquidar (diario): {queries['RF_PLI_007_MontoLiquidar']['Monto a Liquidar'].iloc[0]:,.2f}")

# Ejecutar función principal
flujo_gob_clp = monto_liq_gob_clp(
    df_cartera_mon_total=queries['RF_PLI_003_CarteraGobCLP_MonTotal'],
    df_haircut_dia_pcto=queries['RF_PLI_006c_Haircut_Dia_Pcto'],
    df_monto_liquidar=queries['RF_PLI_007_MontoLiquidar']
)

print("\n" + "="*70)
print("RESULTADO: Flujo_GobCLP")
print("="*70)
print(f"Registros generados: {len(flujo_gob_clp)}")
display(flujo_gob_clp.head(15))
display(flujo_gob_clp.tail(10))

FASE 5: INTEGRACIÓN - Ejecutando monto_liq_gob_clp()

Inputs:
  1. RF_PLI_003_CarteraGobCLP_MonTotal: 1 registros
     VP_Flujo = 626,643,809,581.59
  2. RF_PLI_006c_Haircut_Dia_Pcto: 90 registros
  3. RF_PLI_007_MontoLiquidar: 1 registros
     Monto a Liquidar (diario): 111,015,499,404.70

RESULTADO: Flujo_GobCLP
Registros generados: 91


,Dia,DiaSem,Haircut,Monto_Liquidar
0,0.0,NaN,0.000000,6.266438e+11
1,1.0,5.0,0.004047,1.110155e+11
2,2.0,6.0,0.004205,0.000000e+00
3,3.0,7.0,0.004362,0.000000e+00
4,4.0,1.0,0.004520,1.110155e+11
5,5.0,2.0,0.004679,1.110155e+11
6,6.0,3.0,0.004838,1.110155e+11
7,7.0,4.0,0.004997,1.110155e+11
8,8.0,5.0,0.005254,6.862433e+10
9,9.0,6.0,0.005510,0.000000e+00


,Dia,DiaSem,Haircut,Monto_Liquidar
81,81.0,1.0,0.015046,0.0
82,82.0,2.0,0.015145,0.0
83,83.0,3.0,0.015244,0.0
84,84.0,4.0,0.015342,0.0
85,85.0,5.0,0.015441,0.0
86,86.0,6.0,0.015539,0.0
87,87.0,7.0,0.015638,0.0
88,88.0,1.0,0.015736,0.0
89,89.0,2.0,0.015835,0.0
90,90.0,3.0,0.015933,0.0


In [16]:
# =============================================================================
# EJECUTAR PIPELINE GOBCLF (GOBIERNO CLF - BONOS UF)
# =============================================================================

# Preparar diccionario de tablas necesarias
tablas_gobclf = {
    'RF_FactCLF_Gob': tablas['RF_FactCLF_Gob'],
    'FPL': tablas['FPL'],
    'RF_MontosLiq': tablas['RF_MontosLiq']
}

# Ejecutar pipeline completo
flujo_gob_clf, queries_gobclf = helpers.generar_flujo_liquidacion_instrumento(
    df_cartera_inv=queries['RF_PLI_001_CarteraInv'],
    df_cartera_inv_pacto=queries['RF_PLI_001d_CarteraInv_Pcto'],
    tablas=tablas_gobclf,
    tipo_instrumento='GobCLF',
    fecha_proceso=fecha_proceso,
    verbose=True
)

print("\n" + "="*70)
print("RESUMEN FLUJO GobCLF")
print("="*70)
display(flujo_gob_clf.head(15))


PIPELINE DE LIQUIDACIÓN: Gobierno CLF
Tipo: GobCLF
Códigos disponible: ['BCU', 'BTU']
Códigos pacto: ['BCU', 'BTU', 'CER']
Tabla factores: RF_FactCLF_Gob
Instrumento FPL: Gobierno CLF
Cartera GobCLF: 68 registros después de filtrar por instrumento ['BCU', 'BTU']

Cartera filtrada para GobCLF: 68 registros
Monto total GobCLF generado: 1 registros después de agrupar por ['Fec_Pro', 'Cod_Pro', 'Moneda']

Monto total calculado para GobCLF: 1 registros
Cartera ponderada RF_CarteraGobCLF_Pond: 68 registros generados.

Cartera ponderada para GobCLF: 68 registros
Cartera GobCLF_Pacto: 0 registros después de filtrar por instrumento ['BCU', 'BTU', 'CER']

Cartera pactos filtrada para GobCLF: 0 registros

Monto por plazo de pacto para GobCLF: 0 registros

Usando tabla de factores: RF_FactCLF_Gob (1,350 registros)

FASE 2.1: RF_PLI_0XX_CarteraHC (Cartera con Haircut)
Registros cartera entrada: 68
Registros factores: 1,350

[FPL] Haircut para '['Gobierno CLF']': 0.006937

[JOIN] Registros después 

,Dia,DiaSem,Haircut,Monto_Liquidar
0,0.0,NaN,0.000000,1.094037e+06
1,1.0,5.0,0.006937,7.369081e+05
2,2.0,6.0,0.008281,0.000000e+00
3,3.0,7.0,0.010069,0.000000e+00
4,4.0,1.0,0.011858,3.477801e+05
5,5.0,2.0,0.013647,0.000000e+00
6,6.0,3.0,0.015436,0.000000e+00
7,7.0,4.0,0.017224,0.000000e+00
8,8.0,5.0,0.018647,0.000000e+00
9,9.0,6.0,0.020069,0.000000e+00


In [17]:
# =============================================================================
# EJECUTAR PIPELINE DPF (DEPOSITOS A PLAZO FIJO)
# =============================================================================
# =============================================================================
import importlib
import helpers
importlib.reload(helpers)

# Preparar diccionario de tablas necesarias
tablas_dpf = {
    'RF_FactCLP_Banc': tablas['RF_FactCLP_Banc'],
    'FPL': tablas['FPL'],
    'RF_MontosLiq': tablas['RF_MontosLiq']
}
# Ejecutar pipeline completo
flujo_dpf, queries_dpf = helpers.generar_flujo_liquidacion_instrumento(
    df_cartera_inv=queries['RF_PLI_001_CarteraInv'],
    df_cartera_inv_pacto=queries['RF_PLI_001d_CarteraInv_Pcto'],
    tablas=tablas_dpf,
    tipo_instrumento='DPF',
    fecha_proceso=fecha_proceso,
    verbose=True
)


PIPELINE DE LIQUIDACIÓN: Depósito a Plazo Fijo
Tipo: DPF
Códigos disponible: ['DPF']
Códigos pacto: ['DPF', 'FFM']
Tabla factores: RF_FactCLP_Banc
Instrumento FPL: DPF
Cartera DPF: 6 registros después de filtrar por instrumento ['DPF']

Cartera filtrada para DPF: 6 registros
Monto total DPF generado: 1 registros después de agrupar por ['Fec_Pro', 'Cod_Pro', 'Moneda']

Monto total calculado para DPF: 1 registros
Cartera ponderada RF_CarteraDPF_Pond: 6 registros generados.

Cartera ponderada para DPF: 6 registros
Cartera DPF_Pacto: 0 registros después de filtrar por instrumento ['DPF', 'FFM']

Cartera pactos filtrada para DPF: 0 registros

Monto por plazo de pacto para DPF: 0 registros

Usando tabla de factores: RF_FactCLP_Banc (1,260 registros)

FASE 2.1: RF_PLI_0XX_CarteraHC (Cartera con Haircut)
Registros cartera entrada: 6
Registros factores: 1,260

[FPL] Haircut para '['DPF']': 0.004804

[JOIN] Registros después de JOIN con factores: 540

[CALC] FactorPond calculado usando MAX(Fact

In [18]:
tablas_dpr = {
    'RF_FactCLF_Banc': tablas['RF_FactCLF_Banc'],
    'FPL': tablas['FPL'],
    'RF_MontosLiq': tablas['RF_MontosLiq']
}

# Ejecutar pipeline completo
flujo_dpr, queries_dpr = helpers.generar_flujo_liquidacion_instrumento(
    df_cartera_inv=queries['RF_PLI_001_CarteraInv'],
    df_cartera_inv_pacto=queries['RF_PLI_001d_CarteraInv_Pcto'],
    tablas=tablas_dpr,
    tipo_instrumento='DPR',
    fecha_proceso=fecha_proceso,
    verbose=True
)

tablas_bbc = {
    'RF_FactCLP_Banc': tablas['RF_FactCLP_Banc'],
    'FPL': tablas['FPL'],
    'RF_MontosLiq': tablas['RF_MontosLiq']
}
# Ejecutar pipeline completo
flujo_bbc, queries_bbc = helpers.generar_flujo_liquidacion_instrumento(
    df_cartera_inv=queries['RF_PLI_001_CarteraInv'],
    df_cartera_inv_pacto=queries['RF_PLI_001d_CarteraInv_Pcto'],
    tablas=tablas_bbc,
    tipo_instrumento='BBC',
    fecha_proceso=fecha_proceso,
    verbose=True
)

tablas_lch = {
    'RF_FactCLF_Banc': tablas['RF_FactCLF_Banc'],
    'FPL': tablas['FPL'],
    'RF_MontosLiq': tablas['RF_MontosLiq']
}

# Ejecutar pipeline completo
flujo_lch, queries_lch = helpers.generar_flujo_liquidacion_instrumento(
    df_cartera_inv=queries['RF_PLI_001_CarteraInv'],
    df_cartera_inv_pacto=queries['RF_PLI_001d_CarteraInv_Pcto'],
    tablas=tablas_lch,
    tipo_instrumento='LCH',
    fecha_proceso=fecha_proceso,
    verbose=True
)
# =============================================================================


PIPELINE DE LIQUIDACIÓN: Depósito a Plazo Reajustable
Tipo: DPR
Códigos disponible: ['DPR']
Códigos pacto: ['DPR']
Tabla factores: RF_FactCLF_Banc
Instrumento FPL: DPR
Cartera DPR: 5 registros después de filtrar por instrumento ['DPR']

Cartera filtrada para DPR: 5 registros
Monto total DPR generado: 1 registros después de agrupar por ['Fec_Pro', 'Cod_Pro', 'Moneda']

Monto total calculado para DPR: 1 registros
Cartera ponderada RF_CarteraDPR_Pond: 5 registros generados.

Cartera ponderada para DPR: 5 registros
Cartera DPR_Pacto: 0 registros después de filtrar por instrumento ['DPR']

Cartera pactos filtrada para DPR: 0 registros

Monto por plazo de pacto para DPR: 0 registros

Usando tabla de factores: RF_FactCLF_Banc (1,350 registros)

FASE 2.1: RF_PLI_0XX_CarteraHC (Cartera con Haircut)
Registros cartera entrada: 5
Registros factores: 1,350

[FPL] Haircut para '['DPR']': 0.001980

[JOIN] Registros después de JOIN con factores: 450

[CALC] FactorPond calculado usando MAX(Factor, 0.0

In [19]:
# TODO: agregar DPX (en USD) con los parámetros de DPR, pero con montos a liquidar en dólares. 


In [20]:
# =============================================================================
# COMPARACION DE FLUJOS OBTENIDOS VS FLUJOS EN ACCESS
# =============================================================================

flujos = {
    'GobCLP': flujo_gob_clp,
    'GobCLF': flujo_gob_clf,
    'DPF': flujo_dpf,
    'DPR': flujo_dpr,
    'BBC': flujo_bbc,
    'LCH': flujo_lch
    
    }

# leer tablas access local, que están en .pkl en data_path
with open(data_path / f"tablas_access_prod_20260115_20260119_102556.pkl", "rb") as f:
    tablas_access = pickle.load(f)
flujos_access = {
    'GobCLP': tablas_access['Flujo_GobCLP'],
    'GobCLF': tablas_access['Flujo_GobCLF'],
    'DPF': tablas_access['Flujo_DPF'],
    'DPR': tablas_access['Flujo_DPR'],
    'BBC': tablas_access['Flujo_BBC'],
    'LCH': tablas_access['Flujo_LCH']
    }



In [21]:
# comparar cada flujo generado con el de access

for inst in flujos.keys():
    print("\n" + "="*70)
    print(f"Comparando flujo {inst}")
    print("="*70)
    df_gen = flujos[inst]
    df_acc = flujos_access[inst]
    
    # comparar diferencias por día en Monto_Liquidar
    df_comp = pd.merge(
        df_gen.groupby('Dia').sum().reset_index(),
        df_acc.groupby('Dia').sum().reset_index(),
        on='Dia',
        suffixes=('_gen', '_acc')
    )
    # diferencias en cada columna:
    for col in ['Monto_Liquidar', 'Haircut']:
        df_comp[f'Diff_{col}'] = df_comp[f'{col}_gen'] - df_comp[f'{col}_acc']


    # log de resumen de diferencias
    total_diff = df_comp[[f'Diff_{col}' for col in ['Monto_Liquidar', 'Haircut',]]].sum()
    print(f"Resumen diferencias totales:")
    for col in ['Monto_Liquidar', 'Haircut']:
        print(f"  - {col}: {total_diff[f'Diff_{col}']:}")
    print("="*70)
    print("Días con diferencias significativas (> 1 peso para Monto_Liquidar, > 0.01 para Haircut):")
    diffs_significativos = df_comp[
        (df_comp['Diff_Monto_Liquidar'].abs() > 1) |
        (df_comp['Diff_Haircut'].abs() > 0.0001)
    ]
    if diffs_significativos.empty:
        print("  - No se encontraron diferencias significativas.")
    else:
        display(diffs_significativos)


Comparando flujo GobCLP
Resumen diferencias totales:
  - Monto_Liquidar: -3.0517578125e-05
  - Haircut: -3.8163916471489756e-17
Días con diferencias significativas (> 1 peso para Monto_Liquidar, > 0.01 para Haircut):
  - No se encontraron diferencias significativas.

Comparando flujo GobCLF
Resumen diferencias totales:
  - Monto_Liquidar: -4.656612873077393e-10
  - Haircut: 4.891920202254596e-16
Días con diferencias significativas (> 1 peso para Monto_Liquidar, > 0.01 para Haircut):
  - No se encontraron diferencias significativas.

Comparando flujo DPF
Resumen diferencias totales:
  - Monto_Liquidar: 0.0
  - Haircut: 1.0408340855860843e-17
Días con diferencias significativas (> 1 peso para Monto_Liquidar, > 0.01 para Haircut):
  - No se encontraron diferencias significativas.

Comparando flujo DPR
Resumen diferencias totales:
  - Monto_Liquidar: 0.0
  - Haircut: 8.239936510889834e-18
Días con diferencias significativas (> 1 peso para Monto_Liquidar, > 0.01 para Haircut):
  - No se en

---
# 🏗️ SECCIÓN C: Tabla Final de Inversiones (Pasos 20-21)

## C.1 Paso 20: RF_PLI_045_Gener_Precios_Dia

**Descripción:** Obtiene el precio UF (TCRC) del día para conversión de monedas.

**SQL Original:**
```sql
SELECT Fecha, NEMOTECNICO, Instrumento, Precio_Mid 
INTO Precios_Dia
FROM RF_Base_Diaria_Precios
WHERE Instrumento = "TCRC" AND Fecha = @fecha_proceso
```

**Implementación:** Ya realizada parcialmente en celda anterior.

In [22]:
#print(queries_raw.loc[queries_raw['nombre']=='RF_PLI_045_Gener_Precios_Dia','sql'].values[0])

# RF_Base_Diaria_Precios es una tabla externa, que se llama de la misma forma,
#  en el archivo access '\\\\vmdvorak\\Riesgo Financiero2\\RF_PROCESOS\\RF_Proceso_Tasas\\RF_Base_PT.accdb'
# y que está en el diccionario tablas
columnas_RF_PLI_045_Gener_Precios_Dia = ['Fecha','NEMOTECNICO','Instrumento','Precio_Mid']
filtro_RF_PLI_045_Gener_Precios_Dia = 'Instrumento=="TCRC"'
queries['RF_PLI_045_Gener_Precios_Dia'] = tablas['RF_Base_Diaria_Precios'][columnas_RF_PLI_045_Gener_Precios_Dia].query(
    'Fecha==@fecha_proceso').query(filtro_RF_PLI_045_Gener_Precios_Dia)
queries['RF_PLI_045_Gener_Precios_Dia']


,Fecha,NEMOTECNICO,Instrumento,Precio_Mid
454136,2026-01-15,AUD,TCRC,591.990016
454137,2026-01-15,COP,TCRC,0.239949
454138,2026-01-15,BRL,TCRC,164.697105
454139,2026-01-15,CAD,TCRC,635.463443
454140,2026-01-15,CHF,TCRC,1099.813177
454141,2026-01-15,CLF,TCRC,39747.120000
454142,2026-01-15,CLP,TCRC,1.000000
454143,2026-01-15,CNY,TCRC,126.767923
454144,2026-01-15,ARS,TCRC,0.611261
454145,2026-01-15,DKK,TCRC,137.205364


---
## C.2 Paso 21: RF_PLI_044e_Modelo_Inversiones_Tabla_Final

> 📄 **Ver análisis completo en:** [RF_PLI_044e_analisis.md](./RF_PLI_044e_Modelo_Inversiones_Tabla_Final_analisis.md)

**Descripción:** UNION de 12 queries anidadas que consolida:
- 6 flujos de instrumentos (GobCLP, GobCLF, DPF, DPR, BBC, LCH)
- Cartera de garantías
- Pactos de retrocompra

**Dependencias identificadas:**

```
RF_PLI_001b_CarteraInv_Gtia → RF_PLI_001c_CarteraInv_Gtia
RF_PLI_008b a 043b (*_Final)  →  RF_PLI_044_Modelo_Inversiones_Final
RF_PLI_044c_Pacto_FB          →  RF_PLI_044d_Full
                                    ↓
                              RF_PLI_044e_Tabla_Final
```

In [27]:
df_secuencia.tail(8)


,paso,tipo_accion,nombre_accion,descripcion,query_tipo,linea_vba,codigo_vba,longitud_codigo,codigo_completo,query_hash
19,20,SQL_QUERY,RF_PLI_045_Gener_Precios_Dia,SQL_QUERY: RF_PLI_045_Gener_Precios_Dia,DDL,1481,"DoCmd.OpenQuery ""RF_PLI_045_Gener_Precios_Dia""...",349,"SELECT RF_Base_Diaria_Precios.Fecha, RF_Base_D...",bced1187a6d42d116dbe02c3d82d42c3874b3329
20,21,SQL_QUERY,RF_PLI_044e_Modelo_Inversiones_Tabla_Final,SQL_QUERY: RF_PLI_044e_Modelo_Inversiones_Tabl...,DDL,1482,"DoCmd.OpenQuery ""RF_PLI_044e_Modelo_Inversione...",130,SELECT RF_PLI_044d_Modelo_Inversiones_Full.* I...,e45b24b7630c2b5faa70daff054181cb4a875490
21,22,SQL_QUERY,RF_PLI_047_Limpia_Tabla_Desarrollo_Interna,SQL_QUERY: RF_PLI_047_Limpia_Tabla_Desarrollo_...,Type32,1483,"DoCmd.OpenQuery ""RF_PLI_047_Limpia_Tabla_Desar...",595,"DELETE RF_Tabla_Desarrollo_Interna.Fec_Pro, RF...",8e44dc2a518d828f7210e6d17ceda226b2e4d01e
22,23,SQL_QUERY,RF_PLI_048_Tabla_Desarrollo_Interna_Add_ML,SQL_QUERY: RF_PLI_048_Tabla_Desarrollo_Interna...,Type64,1484,"DoCmd.OpenQuery ""RF_PLI_048_Tabla_Desarrollo_I...",1098,INSERT INTO RF_Tabla_Desarrollo_Interna ( Fec_...,c31acdaf400dc4c5f4942bd3154c68362204661b
23,24,SQL_QUERY,RF_PLI_048a_Tabla_Desarrollo_Interna_Add_FFMM,SQL_QUERY: RF_PLI_048a_Tabla_Desarrollo_Intern...,Type64,1485,"DoCmd.OpenQuery ""RF_PLI_048a_Tabla_Desarrollo_...",1009,INSERT INTO RF_Tabla_Desarrollo_Interna ( Fec_...,0542b055e927b923053ae223bccd7d3674881558
24,25,SQL_QUERY,RF_PLI_048b_Tabla_Desarrollo_Interna_Add_HTM,SQL_QUERY: RF_PLI_048b_Tabla_Desarrollo_Intern...,Type64,1486,"DoCmd.OpenQuery ""RF_PLI_048b_Tabla_Desarrollo_...",989,INSERT INTO RF_Tabla_Desarrollo_Interna ( Fec_...,8b1744445bb7d096d042442ffaa3099baca01e34
25,26,SQL_QUERY,RF_PLI_048c_Tabla_Desarrollo_Interna_Add_RT,SQL_QUERY: RF_PLI_048c_Tabla_Desarrollo_Intern...,Type64,1487,"DoCmd.OpenQuery ""RF_PLI_048c_Tabla_Desarrollo_...",973,INSERT INTO RF_Tabla_Desarrollo_Interna ( Fec_...,6dbac1bc1bafd9139736621409be751d538925bf
26,27,SQL_QUERY,RF_PLI_050_Tabla_Desarrollo_Modelo_Inversiones...,SQL_QUERY: RF_PLI_050_Tabla_Desarrollo_Modelo_...,DDL,1488,"DoCmd.OpenQuery ""RF_PLI_050_Tabla_Desarrollo_M...",2079,SELECT RF_PLI_049_Tabla_Desarrollo_Modelo_Inve...,edd8d07a6a4a21a3031106524fa1771bed63319d


In [37]:
print(queries_raw[queries_raw['nombre'].str.contains('_Monto_FueraPlazo')])

                              nombre    tipo  \
17      RF_PLI_003c_Monto_FueraPlazo  Select   
18  RF_PLI_003c_Monto_FueraPlazo_ori  Select   
31      RF_PLI_010c_Monto_FueraPlazo  Select   
32  RF_PLI_010c_Monto_FueraPlazo_ori  Select   
45      RF_PLI_017c_Monto_FueraPlazo  Select   
46  RF_PLI_017c_Monto_FueraPlazo_ori  Select   
60      RF_PLI_024c_Monto_FueraPlazo  Select   
61  RF_PLI_024c_Monto_FueraPlazo_ori  Select   
74      RF_PLI_031c_Monto_FueraPlazo  Select   
75  RF_PLI_031c_Monto_FueraPlazo_ori  Select   
88      RF_PLI_038c_Monto_FueraPlazo  Select   
89  RF_PLI_038c_Monto_FueraPlazo_ori  Select   

                                                  sql  \
17  SELECT RF_PLI_003b_GobCLP_MontoPlazo_Pacto.Dia...   
18  SELECT RF_PLI_003b_GobCLP_MontoPlazo_Pacto.Dia...   
31  SELECT RF_PLI_010b_GobCLF_MontoPlazo_Pacto.Dia...   
32  SELECT RF_PLI_010b_GobCLF_MontoPlazo_Pacto.Dia...   
45  SELECT RF_PLI_017b_DPF_MontoPlazo_Pacto.Dias_P...   
46  SELECT RF_PLI_017b_DPF_MontoP

In [ ]:
['RF_PLI_044b_Modelo_Inversiones_Pacto_FB','RF_PLI_003c_Monto_FueraPlazo','RF_PLI_010c_Monto_FueraPlazo','RF_PLI_017c_Monto_FueraPlazo',
    'RF_PLI_024c_Monto_FueraPlazo','RF_PLI_031c_Monto_FueraPlazo','RF_PLI_038c_Monto_FueraPlazo']

In [28]:
# Ver los pasos pendientes (21+) de df_secuencia
queries_finales = df_secuencia[df_secuencia['paso']>=20]
print(f"Pasos 20 a {df_secuencia['paso'].max()} de la secuencia:")
display(queries_finales)

Pasos 20 a 27 de la secuencia:


,paso,tipo_accion,nombre_accion,descripcion,query_tipo,linea_vba,codigo_vba,longitud_codigo,codigo_completo,query_hash
19,20,SQL_QUERY,RF_PLI_045_Gener_Precios_Dia,SQL_QUERY: RF_PLI_045_Gener_Precios_Dia,DDL,1481,"DoCmd.OpenQuery ""RF_PLI_045_Gener_Precios_Dia""...",349,"SELECT RF_Base_Diaria_Precios.Fecha, RF_Base_D...",bced1187a6d42d116dbe02c3d82d42c3874b3329
20,21,SQL_QUERY,RF_PLI_044e_Modelo_Inversiones_Tabla_Final,SQL_QUERY: RF_PLI_044e_Modelo_Inversiones_Tabl...,DDL,1482,"DoCmd.OpenQuery ""RF_PLI_044e_Modelo_Inversione...",130,SELECT RF_PLI_044d_Modelo_Inversiones_Full.* I...,e45b24b7630c2b5faa70daff054181cb4a875490
21,22,SQL_QUERY,RF_PLI_047_Limpia_Tabla_Desarrollo_Interna,SQL_QUERY: RF_PLI_047_Limpia_Tabla_Desarrollo_...,Type32,1483,"DoCmd.OpenQuery ""RF_PLI_047_Limpia_Tabla_Desar...",595,"DELETE RF_Tabla_Desarrollo_Interna.Fec_Pro, RF...",8e44dc2a518d828f7210e6d17ceda226b2e4d01e
22,23,SQL_QUERY,RF_PLI_048_Tabla_Desarrollo_Interna_Add_ML,SQL_QUERY: RF_PLI_048_Tabla_Desarrollo_Interna...,Type64,1484,"DoCmd.OpenQuery ""RF_PLI_048_Tabla_Desarrollo_I...",1098,INSERT INTO RF_Tabla_Desarrollo_Interna ( Fec_...,c31acdaf400dc4c5f4942bd3154c68362204661b
23,24,SQL_QUERY,RF_PLI_048a_Tabla_Desarrollo_Interna_Add_FFMM,SQL_QUERY: RF_PLI_048a_Tabla_Desarrollo_Intern...,Type64,1485,"DoCmd.OpenQuery ""RF_PLI_048a_Tabla_Desarrollo_...",1009,INSERT INTO RF_Tabla_Desarrollo_Interna ( Fec_...,0542b055e927b923053ae223bccd7d3674881558
24,25,SQL_QUERY,RF_PLI_048b_Tabla_Desarrollo_Interna_Add_HTM,SQL_QUERY: RF_PLI_048b_Tabla_Desarrollo_Intern...,Type64,1486,"DoCmd.OpenQuery ""RF_PLI_048b_Tabla_Desarrollo_...",989,INSERT INTO RF_Tabla_Desarrollo_Interna ( Fec_...,8b1744445bb7d096d042442ffaa3099baca01e34
25,26,SQL_QUERY,RF_PLI_048c_Tabla_Desarrollo_Interna_Add_RT,SQL_QUERY: RF_PLI_048c_Tabla_Desarrollo_Intern...,Type64,1487,"DoCmd.OpenQuery ""RF_PLI_048c_Tabla_Desarrollo_...",973,INSERT INTO RF_Tabla_Desarrollo_Interna ( Fec_...,6dbac1bc1bafd9139736621409be751d538925bf
26,27,SQL_QUERY,RF_PLI_050_Tabla_Desarrollo_Modelo_Inversiones...,SQL_QUERY: RF_PLI_050_Tabla_Desarrollo_Modelo_...,DDL,1488,"DoCmd.OpenQuery ""RF_PLI_050_Tabla_Desarrollo_M...",2079,SELECT RF_PLI_049_Tabla_Desarrollo_Modelo_Inve...,edd8d07a6a4a21a3031106524fa1771bed63319d


In [47]:
# Ver el SQL de cada query pendiente (pasos 20-27)
print("="*80)
print("QUERIES PENDIENTES - SQL COMPLETO")
print("="*80)

for idx, row in df_secuencia[df_secuencia['paso']>=20].iterrows():
    print(f"\n{'='*80}")
    print(f"PASO {row['paso']}: {row['nombre_accion']}")
    print(f"Tipo: {row['query_tipo']} | Longitud: {row['longitud_codigo']} chars")
    print("-"*80)
    print(row['codigo_completo'])
    print("="*80)

QUERIES PENDIENTES - SQL COMPLETO

PASO 20: RF_PLI_045_Gener_Precios_Dia
Tipo: DDL | Longitud: 349 chars
--------------------------------------------------------------------------------
SELECT RF_Base_Diaria_Precios.Fecha, RF_Base_Diaria_Precios.NEMOTECNICO, RF_Base_Diaria_Precios.Instrumento, RF_Base_Diaria_Precios.Precio_Mid INTO Precios_Dia
FROM RF_Fecha_Proceso_Carteras INNER JOIN RF_Base_Diaria_Precios ON RF_Fecha_Proceso_Carteras.Fecha = RF_Base_Diaria_Precios.Fecha
WHERE (((RF_Base_Diaria_Precios.Instrumento)="TCRC"));


PASO 21: RF_PLI_044e_Modelo_Inversiones_Tabla_Final
Tipo: DDL | Longitud: 130 chars
--------------------------------------------------------------------------------
SELECT RF_PLI_044d_Modelo_Inversiones_Full.* INTO RF_PLI_Modelo_Inversiones_Final_CLP
FROM RF_PLI_044d_Modelo_Inversiones_Full;


PASO 22: RF_PLI_047_Limpia_Tabla_Desarrollo_Interna
Tipo: Type32 | Longitud: 595 chars
--------------------------------------------------------------------------------
DEL

---
# 💾 SECCIÓN D: Integración Tabla Desarrollo (Pasos 22-27)

> 📄 **Ver análisis completo en:** [analisis_pasos_22_27.md](./analisis_pasos_22_27_tabla_desarrollo.md)

## Resumen de Pasos Pendientes

| Paso | Query | Tipo | Descripción | Complejidad |
|------|-------|------|-------------|-------------|
| 22 | RF_PLI_047_Limpia | DELETE | Limpiar tabla destino | ⭐ Trivial (N/A en Python) |
| 23 | RF_PLI_048_Add_ML | INSERT | Insertar flujos modelo liquidación | ⭐⭐ Simple |
| 24 | RF_PLI_048a_Add_FFMM | INSERT | Insertar fondos mutuos | ⭐⭐ Simple |
| 25 | RF_PLI_048b_Add_HTM | INSERT | Insertar held-to-maturity | ⭐⭐ Simple |
| 26 | RF_PLI_048c_Add_RT | INSERT | Insertar renta en tránsito | ⭐⭐ Simple |
| 27 | RF_PLI_050_Excel | SELECT | Formatear para Excel | ⭐⭐ Simple |

## Por Qué FFMM, HTM y RT NO Pasan por el Modelo de Liquidación

- **FFMM (Fondos Mutuos):** Liquidez inmediata, rescate en T+0/T+1
- **HTM (Held-to-Maturity):** Compromiso de no venta hasta vencimiento
- **RT (Renta en Tránsito):** Operaciones ya comprometidas, fecha de pago fija

---
## 🧪 Validación: output/tabla_final.py

Validamos las funciones implementadas contra datos de Access:

In [31]:
# =============================================================================
# 🚧 VALIDACIÓN output/tabla_final.py vs Access
# =============================================================================

# Recargar módulo para ver cambios
import importlib
import RF_Modelo_Inversiones.output.tabla_final as tabla_final_mod
importlib.reload(tabla_final_mod)

# Importar nuevas funciones
from RF_Modelo_Inversiones.output.tabla_final import (
    generar_precios_dia,
    formatear_flujo_instrumento,
    generar_tabla_final_inversiones,
    agregar_precio_y_flujo_clp,
    generar_tabla_desarrollo_completa,
    formatear_para_excel,
    ejecutar_pasos_20_a_27,
    COLUMNAS_TABLA_FINAL,
    COLUMNAS_TABLA_DESARROLLO,
    MAPEO_COLUMNAS_EXCEL
)

print("✅ Funciones importadas correctamente (recargadas)")
print(f"📊 Columnas tabla final: {len(COLUMNAS_TABLA_FINAL)}")
print(f"📊 Columnas tabla desarrollo: {len(COLUMNAS_TABLA_DESARROLLO)}")
print(f"📊 Mapeo Excel: {len(MAPEO_COLUMNAS_EXCEL)} columnas")

✅ Funciones importadas correctamente (recargadas)
📊 Columnas tabla final: 12
📊 Columnas tabla desarrollo: 14
📊 Mapeo Excel: 14 columnas


In [32]:
# =============================================================================
# Paso 20: Generar Precios del Día (TCRC filtrado)
# =============================================================================
# OJO: los precios del día se generan desde la tabla linkeada RF_Base_Diaria_Precios,
# que es una tabla externa en Access, que se encuentra en el archivo:
# '\\\\vmdvorak\\Riesgo Financiero2\\RF_PROCESOS\\RF_Proceso_Tasas\\RF_Base_PT.accdb'
# y que ya está cargada en el diccionario tablas como 'RF_Base_Diaria_Precios'
df_precios = tablas['RF_Base_Diaria_Precios'] 
# Aplicar función para generar precios del día

df_precios_dia = generar_precios_dia(
    df_precios=df_precios,
    fecha_proceso=fecha_proceso,
    verbose=True
)
print(f"\n📊 Precios del día generados: {len(df_precios_dia):,} filas")
display(df_precios_dia.head())



[Paso 20] Generando precios del día...
  Instrumento: TCRC
  ✓ Precio TCRC: 591.9900

📊 Precios del día generados: 23 filas


,Fecha,NEMOTECNICO,Instrumento,Precio_Mid
454136,2026-01-15,AUD,TCRC,591.990016
454137,2026-01-15,COP,TCRC,0.239949
454138,2026-01-15,BRL,TCRC,164.697105
454139,2026-01-15,CAD,TCRC,635.463443
454140,2026-01-15,CHF,TCRC,1099.813177


In [55]:
# =============================================================================
# Paso 21: Generar Tabla Final Inversiones (UNION ALL)
# =============================================================================

# Preparar diccionario de flujos (ya existen en el notebook)
flujos = {
    'GobCLP': flujo_gob_clp,
    'GobCLF': flujo_gob_clf, 
    'DPF': flujo_dpf,
    'DPR': flujo_dpr,
    'BBC': flujo_bbc,
    'LCH': flujo_lch
}

# Cargar cartera base para garantías (RF_PLI_001b)
df_base = tablas_access.get('RF_PLI_001b_TblCartera_RFBFA', pd.DataFrame())
print(f"📋 Cartera base (para garantías): {len(df_base):,} filas")

# Cargar pactos FB
df_pactos = tablas_access.get('RF_PLI_044c_Add_Pactos_FB', pd.DataFrame())
print(f"📋 Pactos FB: {len(df_pactos):,} filas")

# Generar tabla final
df_tabla_final = generar_tabla_final_inversiones(
    flujos=flujos,
    fecha_proceso=fecha_proceso,
    df_base=df_base,
    df_pactos=df_pactos,
    verbose=True
)

print(f"\n📊 Tabla Final Inversiones: {len(df_tabla_final):,} filas")
print(f"📊 Columnas: {list(df_tabla_final.columns)}")
df_tabla_final.head()

📋 Cartera base (para garantías): 0 filas
📋 Pactos FB: 0 filas
[Paso 21] Generando tabla final de inversiones

  Procesando flujos de instrumentos:
    ✓ ML_C46_Inversiones_Financieras_GOBCLP: 6 flujos, total=623,701,831,416
    ✓ ML_C46_Inversiones_Financieras_GOBCLF: 2 flujos, total=1,084,688
    ✓ ML_C46_Inversiones_Financieras_DPFCLP: 1 flujos, total=25,992,461,906
    ✓ ML_C46_Inversiones_Financieras_DPRCLF: 5 flujos, total=523,605
    ✓ ML_C46_Inversiones_Financieras_CORPCLP: 64 flujos, total=108,127,396,636
    ✓ ML_C46_Inversiones_Financieras_LCHR: 4 flujos, total=1,569,972

  Generando cartera de garantías...
    ⚠ Sin datos de cartera base

  Generando cartera de pactos...
    ⚠ Sin pactos en cartera

  RESUMEN TABLA FINAL:
  - Total registros: 82
  - Total Cap_Amort: 757,824,868,223
  - Columnas: ['Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_A_P', 'Cod_Pro', 'Cod_Sub_Pro', 'Fec_Pago', 'Dias_Pago', 'Cap_Amort', 'Int_Total_Cont', 'VP_Cap_Amort', 'VP_Int_Total_Cont']

📊 Tabla Final Inve

,Fec_Pro,Cod_Emp,Moneda,Cod_A_P,Cod_Pro,Cod_Sub_Pro,Fec_Pago,Dias_Pago,Cap_Amort,Int_Total_Cont,VP_Cap_Amort,VP_Int_Total_Cont
0,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-16,1.0,1.110155e+11,0,1.110155e+11,0
1,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-19,4.0,1.110155e+11,0,1.110155e+11,0
2,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-20,5.0,1.110155e+11,0,1.110155e+11,0
3,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-21,6.0,1.110155e+11,0,1.110155e+11,0
4,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-22,7.0,1.110155e+11,0,1.110155e+11,0


In [57]:
# =============================================================================
# Paso 23: Agregar Precio y Flujo CLP
# =============================================================================

# Aplicar precios y conversión CLF→CLP
df_con_precio = agregar_precio_y_flujo_clp(
    df_inversiones=df_tabla_final,
    df_precios_dia=df_precios_dia,
    verbose=True
)

print(f"\n📊 Tabla con precios: {len(df_con_precio):,} filas")
print(f"📊 Columnas agregadas: Precio_Mid, Flujo_CLP")

# Verificar conversión CLF→CLP
clf_rows = df_con_precio[df_con_precio['Moneda'] == 'CLF']
if len(clf_rows) > 0:
    print(f"\n🔄 Filas CLF procesadas: {len(clf_rows):,}")
    print("   Ejemplo conversión CLF→CLP:")
    ejemplo = clf_rows[['Cod_Sub_Pro', 'Cap_Amort', 'Precio_Mid', 'Flujo_CLP']].head(3)
    display(ejemplo)
else:
    print("\n⚠️ No hay filas CLF en la tabla final")

  ✓ Precio UF aplicado: 591.9900
  ✓ Flujo_CLP total: 759,703,191,129

📊 Tabla con precios: 82 filas
📊 Columnas agregadas: Precio_Mid, Flujo_CLP

🔄 Filas CLF procesadas: 11
   Ejemplo conversión CLF→CLP:


,Cod_Sub_Pro,Cap_Amort,Precio_Mid,Flujo_CLP
6,ML_C46_Inversiones_Financieras_GOBCLF,736908.127935,591.990016,4.362423e+08
7,ML_C46_Inversiones_Financieras_GOBCLF,347780.118340,591.990016,2.058824e+08
9,ML_C46_Inversiones_Financieras_DPRCLF,107876.085527,591.990016,6.386157e+07


In [59]:
# =============================================================================
# Pasos 24-26: Extraer FFMM, HTM, RT (no pasan por liquidación)
# =============================================================================

# Estas carteras se extraen directamente de RF_PLI_001b sin pasar por el modelo de liquidación
# En esta prueba no tenemos cartera base, así que solo usamos los flujos del modelo

df_tabla_desarrollo = generar_tabla_desarrollo_completa(
    df_modelo_inversiones=df_tabla_final,
    df_precios_dia=df_precios_dia,
    df_cartera_ffmm=None,  # Se extraería de df_base si estuviera disponible
    df_cartera_htm=None,
    df_cartera_rt=None,
    verbose=True
)

print(f"\n📊 Tabla Desarrollo Completa: {len(df_tabla_desarrollo):,} filas")

# Verificar columnas
print(f"📊 Columnas: {list(df_tabla_desarrollo.columns)}")


[Pasos 22-26] Generando tabla de desarrollo completa

  [23] Procesando modelo de liquidación...
  ✓ Precio UF aplicado: 591.9900
  ✓ Flujo_CLP total: 759,703,191,129
       ML: 82 registros

  RESUMEN TABLA DESARROLLO:
  - Total registros: 82
  - Total Flujo_CLP: 759,703,191,129

📊 Tabla Desarrollo Completa: 82 filas
📊 Columnas: ['Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_A_P', 'Cod_Pro', 'Cod_Sub_Pro', 'Fec_Pago', 'Dias_Pago', 'Cap_Amort', 'Int_Total_Cont', 'VP_Cap_Amort', 'VP_Int_Total_Cont', 'Precio_Mid', 'Flujo_CLP']


In [61]:
# =============================================================================
# Paso 27: Formatear para Excel
# =============================================================================

df_excel = formatear_para_excel(
    df_tabla_desarrollo=df_tabla_desarrollo,
    verbose=True
)

print(f"\n📊 Formato Excel: {len(df_excel):,} filas x {len(df_excel.columns)} columnas")
print(f"📊 Columnas Excel: {list(df_excel.columns)}")

# Comparar con salida de Access si existe
if 'RF_PLI_050_Gen_Excel_Output' in tablas_access:
    df_excel_access = tablas_access['RF_PLI_050_Gen_Excel_Output']
    print(f"\n📊 Comparación con Access:")
    print(f"   Python: {len(df_excel):,} filas")
    print(f"   Access: {len(df_excel_access):,} filas")
else:
    print("\n⚠️ Tabla RF_PLI_050 no disponible para comparación")

df_excel.head()


[Paso 27] Formateando para Excel...
  ✓ Formateado: 82 registros, 17 columnas

📊 Formato Excel: 82 filas x 17 columnas
📊 Columnas Excel: ['FECHA PROCESO', 'CODIGO_EMPRESA', 'OPERACION', 'COD ACT/PAS', 'MONEDA_ORIGEN', 'MONEDA_COMPENSACION', 'COMPENSACION', 'COD_PRO', 'COD_SUB_PRO', 'FECHA DE PAGO', 'PLAZO_PAGO', 'FLUJO_CAPITAL', 'FLUJO_INTERES', 'VP_CAP', 'VP_INT_CONT', 'PRECIO_MID', 'FLUJO_CLP']

⚠️ Tabla RF_PLI_050 no disponible para comparación


,FECHA PROCESO,CODIGO_EMPRESA,OPERACION,COD ACT/PAS,MONEDA_ORIGEN,MONEDA_COMPENSACION,COMPENSACION,COD_PRO,COD_SUB_PRO,FECHA DE PAGO,PLAZO_PAGO,FLUJO_CAPITAL,FLUJO_INTERES,VP_CAP,VP_INT_CONT,PRECIO_MID,FLUJO_CLP
0,2026-01-15,1,INVERSIONES,ACT,CLP,CLP,NO,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-16,1.0,1.110155e+11,0,1.110155e+11,0,591.990016,1.110155e+11
1,2026-01-15,1,INVERSIONES,ACT,CLP,CLP,NO,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-19,4.0,1.110155e+11,0,1.110155e+11,0,591.990016,1.110155e+11
2,2026-01-15,1,INVERSIONES,ACT,CLP,CLP,NO,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-20,5.0,1.110155e+11,0,1.110155e+11,0,591.990016,1.110155e+11
3,2026-01-15,1,INVERSIONES,ACT,CLP,CLP,NO,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-21,6.0,1.110155e+11,0,1.110155e+11,0,591.990016,1.110155e+11
4,2026-01-15,1,INVERSIONES,ACT,CLP,CLP,NO,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-22,7.0,1.110155e+11,0,1.110155e+11,0,591.990016,1.110155e+11


In [62]:
# =============================================================================
# 🎯 Función Completa: ejecutar_pasos_20_a_27()
# =============================================================================

# Preparar diccionario de tablas (simulando lo que tendríamos en producción)
tablas_input = {
    'RF_Base_Diaria_Precios': df_precios_dia if 'df_precios_dia' in dir() else tablas_access.get('Precios_Dia', pd.DataFrame()),
    'RF_base_Completa_Hist': df_base,  # Para garantías
}

# Ejecutar pipeline completo en una sola llamada
resultado = ejecutar_pasos_20_a_27(
    flujos=flujos,
    tablas=tablas_input,
    fecha_proceso=fecha_proceso,
    df_pactos=df_pactos,
    verbose=True
)

print("\n" + "="*60)
print("📦 RESULTADO FINAL")
print("="*60)
print(f"✅ Precios día:      {len(resultado['precios_dia']):,} filas")
print(f"✅ Tabla final:      {len(resultado['tabla_final_inversiones']):,} filas")  
print(f"✅ Tabla desarrollo: {len(resultado['tabla_desarrollo']):,} filas")
print(f"✅ Formato Excel:    {len(resultado['tabla_excel']):,} filas")


[Paso 20] Generando precios del día...
  Instrumento: TCRC
  ✓ Precio TCRC: 591.9900
[Paso 21] Generando tabla final de inversiones

  Procesando flujos de instrumentos:
    ✓ ML_C46_Inversiones_Financieras_GOBCLP: 6 flujos, total=623,701,831,416
    ✓ ML_C46_Inversiones_Financieras_GOBCLF: 2 flujos, total=1,084,688
    ✓ ML_C46_Inversiones_Financieras_DPFCLP: 1 flujos, total=25,992,461,906
    ✓ ML_C46_Inversiones_Financieras_DPRCLF: 5 flujos, total=523,605
    ✓ ML_C46_Inversiones_Financieras_CORPCLP: 64 flujos, total=108,127,396,636
    ✓ ML_C46_Inversiones_Financieras_LCHR: 4 flujos, total=1,569,972

  Generando cartera de garantías...
    ⚠ Sin datos de cartera base

  Generando cartera de pactos...
    ⚠ Sin pactos en cartera

  RESUMEN TABLA FINAL:
  - Total registros: 82
  - Total Cap_Amort: 757,824,868,223
  - Columnas: ['Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_A_P', 'Cod_Pro', 'Cod_Sub_Pro', 'Fec_Pago', 'Dias_Pago', 'Cap_Amort', 'Int_Total_Cont', 'VP_Cap_Amort', 'VP_Int_Total_Co

---
## 📊 Resumen de Validación

| Paso | Función | Estado | Comparación |
|------|---------|--------|-------------|
| 20 | `generar_precios_dia()` | ✅ | Filtrar TCRC por fecha |
| 21 | `generar_tabla_final_inversiones()` | ✅ | UNION ALL 6 instrumentos + Gtia + Pactos |
| 22 | N/A | ⏭️ | DELETE en Access = crear nuevo DF en Python |
| 23 | `agregar_precio_y_flujo_clp()` | ✅ | JOIN con precios, CLF→CLP |
| 24-26 | `generar_tabla_desarrollo_completa()` | ✅ | Extraer FFMM, HTM, RT |
| 27 | `formatear_para_excel()` | ✅ | Renombrar columnas para export |

---
### 📈 Siguiente: Comparación Numérica vs Access

Cargar datos de Access para validar diferencias < 1 peso:

In [ ]:
# =============================================================================
# 🔍 Comparación Detallada vs Access
# =============================================================================

# Listar tablas de Access disponibles que coinciden con pasos 20-27
tablas_relevantes = [
    'RF_PLI_045_Gener_Precios_Dia',  # Paso 20
    'RF_PLI_046_Tabla_Final_Inversiones',  # Paso 21
    'RF_PLI_048_Tabla_Desarrollo_Interna',  # Paso 23
    'RF_PLI_050_Gen_Excel_Output'  # Paso 27
]

print("📋 Tablas de Access disponibles para comparación:")
for tabla in tablas_relevantes:
    if tabla in tablas_access:
        df_acc = tablas_access[tabla]
        print(f"   ✅ {tabla}: {len(df_acc):,} filas")
    else:
        print(f"   ❌ {tabla}: No disponible")

# Si tenemos la tabla final, hacer comparación numérica
if 'RF_PLI_046_Tabla_Final_Inversiones' in tablas_access:
    df_acc = tablas_access['RF_PLI_046_Tabla_Final_Inversiones']
    print(f"\n📊 Comparación Tabla Final:")
    print(f"   Python: {len(df_tabla_final):,} filas")
    print(f"   Access: {len(df_acc):,} filas")
    
    # Comparar totales numéricos si existe Cap_Amort
    if 'Cap_Amort' in df_tabla_final.columns and 'Cap_Amort' in df_acc.columns:
        total_py = df_tabla_final['Cap_Amort'].sum()
        total_acc = df_acc['Cap_Amort'].sum()
        diff = abs(total_py - total_acc)
        print(f"\n💰 Cap_Amort Total:")
        print(f"   Python: ${total_py:,.0f}")
        print(f"   Access: ${total_acc:,.0f}")
        print(f"   Diferencia: ${diff:,.0f}")
        print(f"   ✅ OK" if diff < 1 else f"   ⚠️ Diferencia > 1 peso")